# DS4 Phenotyping -- Stage 4 Step 2 (Phase 1)

**Notebook:** `59_ds4_phenotype_exploration.ipynb`  
**Prerequisites:** all 5 DS4 Step 1 families -- complete.  
**Blueprint:** `outputs/reports/stage4/step2_phenotyping/ds4_phenotyping_blueprint.md`

---

## Goal

Phase 1 only -- a single curated-feature clustering pass, one feature per model family (5 total), redundancy-checked. Real cross-model complete-case N=641 (verified before writing the blueprint) justifies this scale; further paths are deferred pending what this pass actually finds, not planned in advance by analogy to DS5's six-path outcome.

## Not the goal

- No condition-stratified or friction-response paths -- DS4 has no experimental condition (unlike DS5's Path E/B).
- No commitment to further paths -- that decision is made after seeing this notebook's real result, per the blueprint.

## Output

```
outputs/model_outputs/ds4_phenotyping/
  ds4_phenotype_feature_matrix.csv
  ds4_cluster_assignments.csv
  ds4_cluster_profiles.csv
  ds4_feature_redundancy_report.csv
  ds4_demographic_overlay.csv
  model_run_summary.md
  figures/
```


## FLAG decisions

| Flag | Decision |
|---|---|
| FLAG 1 | Phase 1 only -- further paths deferred pending this result |
| FLAG 2 | Standardize all 5 features (z-score) before clustering |
| FLAG 3 | K-means, silhouette-based k selection (k=2..6) |
| FLAG 4 | Bootstrap Jaccard stability (100 resamples), STABLE threshold >=0.75 |
| FLAG 5 | Demographic overlay exploratory/secondary (N=283 subset, no Likert scales) |


## Features -- one per model family, redundancy-checked

| Family | Feature |
|---|---|
| StoP (53) | `tau_hat` -- relaxation time (days) |
| InT (52) | `H_emp` -- empirical binned entropy of X_t |
| DyS (55) | `fixed_point_deviation` -- OU model-fit quality |
| StM (51) | `chi_OU` -- susceptibility |
| NeT (54) | `rho_self` -- self-transition rate (attempts-only) |

Pre-registered correlation matrix (N=641, verified before writing the blueprint) is reproduced live in cell 7 -- no pair exceeded \|r\|=0.49 in the pre-check.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from scipy import stats

OUT_DIR = Path("outputs/model_outputs/ds4_phenotyping")
FIG_DIR = OUT_DIR / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

FEATURES = ["tau_hat", "H_emp", "fixed_point_deviation", "chi_OU", "rho_self"]
RANDOM_STATE = 42


In [ ]:
stop = pd.read_csv("outputs/model_outputs/ds4_stochastic_processes/participant_parameters.csv")
ent = pd.read_csv("outputs/model_outputs/ds4_information_theory/participant_entropy_metrics.csv")
dys = pd.read_csv("outputs/model_outputs/ds4_dynamical_systems/participant_dynamics_parameters.csv")
stm = pd.read_csv("outputs/model_outputs/ds4_statistical_mechanics/participant_ds4_stm_parameters.csv")
net = pd.read_csv("outputs/model_outputs/ds4_network_models/participant_network_metrics.csv")

df = stop[["participant_id_canonical", "fit_valid", "tau_hat"]].rename(
    columns={"fit_valid": "stop_valid"})
df = df.merge(ent[["participant_id_canonical", "H_emp"]], on="participant_id_canonical", how="left")
df = df.merge(dys[["participant_id_canonical", "fixed_point_deviation"]],
    on="participant_id_canonical", how="left")
df = df.merge(stm[["participant_id_canonical", "chi_OU", "fit_valid"]].rename(
    columns={"fit_valid": "stm_valid"}), on="participant_id_canonical", how="left")
df = df.merge(net[["participant_id_canonical", "rho_self", "eligible"]].rename(
    columns={"eligible": "net_eligible"}), on="participant_id_canonical", how="left")

complete_mask = (
    df["stop_valid"].fillna(False) & df["stm_valid"].fillna(False) &
    df["net_eligible"].fillna(False) & df[FEATURES].notna().all(axis=1)
)
complete = df[complete_mask].reset_index(drop=True)
print(f"Complete-case N: {len(complete)}")


In [ ]:
scaler = StandardScaler()
X = scaler.fit_transform(complete[FEATURES])
print(pd.DataFrame(X, columns=FEATURES).describe())


In [ ]:
corr_matrix = complete[FEATURES].corr()
print(corr_matrix.round(3))
print(f"\nmax |r| off-diagonal: "
      f"{(corr_matrix.values - np.eye(len(FEATURES))).__abs__().max():.3f}")


In [ ]:
def jaccard_cluster_stability(X, k, n_boot=100, random_state=RANDOM_STATE):
    rng = np.random.RandomState(random_state)
    n = X.shape[0]
    km_ref = KMeans(n_clusters=k, n_init=10, random_state=random_state).fit(X)
    ref_labels = km_ref.labels_
    jaccards = []
    for _ in range(n_boot):
        idx = rng.choice(n, size=n, replace=True)
        X_boot = X[idx]
        seed = int(rng.randint(0, 1_000_000))
        km_boot = KMeans(n_clusters=k, n_init=10, random_state=seed).fit(X_boot)
        boot_labels_on_full = km_boot.predict(X)
        best_jaccards = []
        for c in range(k):
            ref_set = set(np.where(ref_labels == c)[0])
            best_j = 0.0
            for c2 in range(k):
                boot_set = set(np.where(boot_labels_on_full == c2)[0])
                union = len(ref_set | boot_set)
                if union == 0:
                    continue
                j = len(ref_set & boot_set) / union
                best_j = max(best_j, j)
            best_jaccards.append(best_j)
        jaccards.append(np.mean(best_jaccards))
    return np.array(jaccards)


# Compute BOTH silhouette and bootstrap stability for every candidate k --
# do not just take the silhouette-argmax: this project's established practice
# (DS5, Structured) is to prefer a stable solution over a raw silhouette-maximal
# one when they disagree, and report the full picture, not just the winner.
k_candidates = range(2, 7)
sil_scores = {}
jaccard_by_k = {}
sizes_by_k = {}
for k in k_candidates:
    labels = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit_predict(X)
    sil_scores[k] = silhouette_score(X, labels)
    jaccard_by_k[k] = float(jaccard_cluster_stability(X, k, n_boot=100).mean())
    sizes_by_k[k] = pd.Series(labels).value_counts().sort_index().tolist()
    verdict = "STABLE" if jaccard_by_k[k] >= 0.75 else "not stable"
    print(f"k={k}: silhouette={sil_scores[k]:.4f}, bootstrap_jaccard="
          f"{jaccard_by_k[k]:.4f} ({verdict}), sizes={sizes_by_k[k]}")

stable_ks = [k for k in k_candidates if jaccard_by_k[k] >= 0.75]
print(f"\nSTABLE candidates: {stable_ks}")
if stable_ks:
    K_CHOSEN = max(stable_ks, key=lambda k: sil_scores[k])
    print(f"Chosen k={K_CHOSEN} (highest silhouette among STABLE candidates)")
else:
    K_CHOSEN = max(sil_scores, key=sil_scores.get)
    print(f"No stable candidate -- falling back to silhouette-argmax k={K_CHOSEN}, "
          f"flag as unstable in the writeup")


In [ ]:
km_final = KMeans(n_clusters=K_CHOSEN, n_init=10, random_state=RANDOM_STATE)
labels_final = km_final.fit_predict(X)
complete["cluster"] = labels_final
print(pd.Series(labels_final).value_counts().sort_index())

mean_jaccard = jaccard_by_k[K_CHOSEN]
stability_verdict = "STABLE" if mean_jaccard >= 0.75 else "NOT STABLE"
jaccard_scores = jaccard_cluster_stability(X, K_CHOSEN, n_boot=100)  # for fig 4's histogram
print(f"\nFinal: k={K_CHOSEN}, bootstrap Jaccard mean={mean_jaccard:.4f} -- {stability_verdict}")


In [ ]:
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import adjusted_rand_score

hier_labels = AgglomerativeClustering(n_clusters=K_CHOSEN, linkage="ward").fit_predict(X)
ari_kmeans_hier = adjusted_rand_score(labels_final, hier_labels)
print(f"Hierarchical clustering (ward, k={K_CHOSEN}) vs k-means: ARI={ari_kmeans_hier:.4f}")
print("Triangulation check: a high ARI means the k-means solution is not an artifact "
      "of that one algorithm -- matches DS5 Phase 1's own method-triangulation practice "
      "(notebook 100, section E: k-means + hierarchical + GMM).")


In [ ]:
cluster_profiles_full = complete.groupby("cluster")[FEATURES].agg(["mean", "median", "std", "count"])
print(cluster_profiles_full)


In [ ]:
pca = PCA(random_state=RANDOM_STATE)
pca_scores = pca.fit_transform(X)
explained = pca.explained_variance_ratio_
print("Explained variance ratio:", np.round(explained, 4))
print(f"PC1+PC2: {explained[0]+explained[1]:.1%}")
complete["PC1"] = pca_scores[:, 0]
complete["PC2"] = pca_scores[:, 1]

pca_loadings = pd.DataFrame(
    pca.components_[:2].T, index=FEATURES, columns=["PC1_loading", "PC2_loading"]
)
print("\nPCA loadings (which features drive PC1/PC2):")
print(pca_loadings.round(3))


In [ ]:
survey = pd.read_csv("outputs/features/ds4/ds4_survey_features.csv")
overlay = complete.merge(
    survey[["participant_id_canonical", "survey_age", "gender_clean"]],
    on="participant_id_canonical", how="left",
)
overlay_valid = overlay.dropna(subset=["survey_age"])
print(f"Demographic overlay N={len(overlay_valid)}")
print(overlay_valid.groupby("cluster")["survey_age"].describe())

groups = [g["survey_age"].dropna().values for _, g in overlay_valid.groupby("cluster")]
groups = [g for g in groups if len(g) > 1]
if len(groups) >= 2:
    f_stat, p_val = stats.f_oneway(*groups)
    print(f"\nANOVA age by cluster: F={f_stat:.3f}, p={p_val:.4f}")
else:
    f_stat, p_val = np.nan, np.nan
    print("\nToo few groups with >1 observation for ANOVA")


In [ ]:
feature_matrix = complete[["participant_id_canonical"] + FEATURES + ["cluster"]].copy()
feature_matrix.to_csv(OUT_DIR / "ds4_phenotype_feature_matrix.csv", index=False)

cluster_assignments = complete[["participant_id_canonical", "cluster"]].copy()
cluster_assignments.to_csv(OUT_DIR / "ds4_cluster_assignments.csv", index=False)

profiles_export = complete.groupby("cluster")[FEATURES].mean().reset_index()
profiles_export["n"] = complete.groupby("cluster").size().values
profiles_export.to_csv(OUT_DIR / "ds4_cluster_profiles.csv", index=False)

print(f"Wrote feature matrix ({len(feature_matrix)} rows), cluster assignments, "
      f"cluster profiles ({len(profiles_export)} clusters)")


In [ ]:
corr_matrix.to_csv(OUT_DIR / "ds4_feature_redundancy_report.csv")
print("Wrote ds4_feature_redundancy_report.csv")

if len(overlay_valid) > 0:
    demo_export = overlay_valid[["participant_id_canonical", "cluster", "survey_age", "gender_clean"]]
    demo_export.to_csv(OUT_DIR / "ds4_demographic_overlay.csv", index=False)
    print(f"Wrote ds4_demographic_overlay.csv ({len(demo_export)} rows)")

summary_lines = []
summary_lines.append("# DS4 Phenotyping -- Model Run Summary")
summary_lines.append("")
summary_lines.append("**Notebook:** `59_ds4_phenotype_exploration.ipynb`  ")
summary_lines.append("**Blueprint:** `outputs/reports/stage4/step2_phenotyping/ds4_phenotyping_blueprint.md`  ")
summary_lines.append("**Run date:** " + pd.Timestamp.now().strftime("%Y-%m-%d"))
summary_lines.append("")
summary_lines.append(f"## Complete-case N: {len(complete)}")
summary_lines.append("")
summary_lines.append("## K-selection -- silhouette AND bootstrap stability for every candidate")
summary_lines.append("")
summary_lines.append("```")
for k in k_candidates:
    verdict = "STABLE" if jaccard_by_k[k] >= 0.75 else "not stable"
    summary_lines.append(f"k={k}: silhouette={sil_scores[k]:.4f}, "
                         f"bootstrap_jaccard={jaccard_by_k[k]:.4f} ({verdict}), "
                         f"sizes={sizes_by_k[k]}")
summary_lines.append("```")
summary_lines.append(f"\nChosen k={K_CHOSEN} -- highest silhouette among STABLE "
                     "candidates (not the raw silhouette-argmax, which was unstable "
                     "here). See blueprint SS1 for this project's established "
                     "stability-first selection practice." if stable_ks else
                     f"\nChosen k={K_CHOSEN} -- WARNING: no candidate k was bootstrap-"
                     "stable; this is the silhouette-argmax fallback, flagged unstable.")
summary_lines.append("")
summary_lines.append("## Stability")
summary_lines.append("")
summary_lines.append(f"Bootstrap Jaccard (100 resamples): mean={mean_jaccard:.4f} -- "
                     f"{stability_verdict}")
summary_lines.append("")
summary_lines.append("## Cluster profiles (feature means)")
summary_lines.append("")
summary_lines.append("```")
summary_lines.append(profiles_export.to_string())
summary_lines.append("```")
summary_lines.append("")
summary_lines.append("## PCA")
summary_lines.append("")
summary_lines.append(f"PC1+PC2 explained variance: {explained[0]+explained[1]:.1%}")
summary_lines.append("")
summary_lines.append("## Demographic overlay (exploratory)")
summary_lines.append("")
summary_lines.append(f"N={len(overlay_valid)}; ANOVA age by cluster: F={f_stat:.3f}, "
                     f"p={p_val:.4f}" if len(groups) >= 2 else "Not enough groups for ANOVA")
summary_lines.append("")
summary_lines.append("## Hierarchical clustering triangulation")
summary_lines.append("")
summary_lines.append(f"ARI(k-means, hierarchical/ward, k={K_CHOSEN}) = {ari_kmeans_hier:.4f}")
summary_lines.append("")
summary_lines.append("## PCA loadings")
summary_lines.append("")
summary_lines.append("```")
summary_lines.append(pca_loadings.round(3).to_string())
summary_lines.append("```")

with open(OUT_DIR / "model_run_summary.md", "w") as f:
    f.write("\n".join(summary_lines) + "\n")
print("Wrote model_run_summary.md")


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ks = list(sil_scores.keys())
vals = [sil_scores[k] for k in ks]
ax.plot(ks, vals, marker="o", color="#4C72B0")
ax.axvline(K_CHOSEN, color="gray", linestyle="--", linewidth=1)
ax.set_xlabel("k")
ax.set_ylabel("silhouette score")
ax.set_title(f"DS4 Phenotyping: silhouette by k (chosen k={K_CHOSEN})")
fig.tight_layout()
fig.savefig(FIG_DIR / "fig1_ds4_silhouette_by_k.png", dpi=150)
plt.close(fig)


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(FEATURES))
width = 0.8 / K_CHOSEN
colors = plt.cm.viridis(np.linspace(0, 1, K_CHOSEN))
for c in range(K_CHOSEN):
    row = profiles_export[profiles_export["cluster"] == c][FEATURES].values.flatten()
    # z-score the profile means for comparability across differently-scaled features
    row_z = (row - complete[FEATURES].mean().values) / complete[FEATURES].std().values
    n_c = int(profiles_export.loc[profiles_export["cluster"] == c, "n"].iloc[0])
    ax.bar(x + c * width, row_z, width=width, color=colors[c], label=f"cluster {c} (n={n_c})")
ax.set_xticks(x + width * (K_CHOSEN - 1) / 2)
ax.set_xticklabels(FEATURES, rotation=20)
ax.set_ylabel("z-scored feature mean")
ax.set_title("DS4 Phenotyping: cluster profiles")
ax.legend(fontsize=8)
ax.axhline(0, color="black", linewidth=0.5)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig2_ds4_cluster_profiles.png", dpi=150)
plt.close(fig)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
sc = ax.scatter(complete["PC1"], complete["PC2"], c=complete["cluster"], cmap="viridis",
                  s=18, alpha=0.7)
ax.set_xlabel(f"PC1 ({explained[0]:.1%})")
ax.set_ylabel(f"PC2 ({explained[1]:.1%})")
ax.set_title("DS4 Phenotyping: PCA projection, colored by cluster")
legend1 = ax.legend(*sc.legend_elements(), title="cluster", fontsize=8)
ax.add_artist(legend1)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig3_ds4_pca_scatter.png", dpi=150)
plt.close(fig)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.hist(jaccard_scores, bins=30, color="#55A868", edgecolor="white")
ax.axvline(0.75, color="black", linestyle="--", linewidth=1, label="STABLE threshold (0.75)")
ax.axvline(mean_jaccard, color="#C44E52", linestyle="-", linewidth=1.5, label=f"mean ({mean_jaccard:.3f})")
ax.set_xlabel("mean Jaccard index (per bootstrap resample)")
ax.set_ylabel("count")
ax.set_title(f"DS4 Phenotyping: bootstrap stability (k={K_CHOSEN}, 100 resamples)")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig4_ds4_bootstrap_jaccard.png", dpi=150)
plt.close(fig)


In [ ]:
from collections import OrderedDict
import matplotlib.patches as mpatches

RADAR_DIMS = OrderedDict([
    ("Relaxation\ntime", "tau_hat"),          # StoP
    ("Behavioral\nentropy", "H_emp"),          # InT
    ("Model-fit\ndeviation", "fixed_point_deviation"),  # DyS
    ("Susceptibility", "chi_OU"),              # StM
    ("Self-\ntransition", "rho_self"),          # NeT
])
AXIS_MODEL = {
    "Relaxation\ntime": "StoP", "Behavioral\nentropy": "InT",
    "Model-fit\ndeviation": "DyS", "Susceptibility": "StM",
    "Self-\ntransition": "NeT",
}
MODEL_COLORS = {
    "InT": "#2196F3", "StoP": "#FF9800", "DyS": "#9C27B0",
    "NeT": "#4CAF50", "StM": "#C5D307",
}
SECTION_LAYERS = [
    {"name": "Behavioral dynamics", "dim_idx": [0, 1, 2], "color": "#C62828"},
    {"name": "Thermodynamic", "dim_idx": [3], "color": "#0A4B96"},
    {"name": "Network structure", "dim_idx": [4], "color": "#044B07"},
]
CLUSTER_COLORS = {0: "#A20A7C", 1: "#7E57C2", 2: "#00897B", 3: "#EF6C00", 4: "#3949AB"}

# Font sizes matched to notebooks/100_phenotype_exploration.ipynb cell 26 (the
# source of the DS5 reference figure, Figure 4.12) -- see
# scripts/regenerate_ds4_phenotype_radar.py, which is the actual script that
# produced the current fig5_ds4_phenotype_radar.png (loads the frozen
# ds4_phenotype_feature_matrix.csv rather than re-running the clustering above).
AXIS_LABEL_FONTSIZE = 18
GRID_TICK_FONTSIZE = 15


def normalize_features(df, dim_feats):
    norm = {}
    for feat in dim_feats:
        v = df[feat].dropna()
        norm[feat] = (df[feat] - v.min()) / (v.max() - v.min() + 1e-12)
    return norm


def draw_model_arcs(ax, angles, dim_labels, axis_model, model_colors,
                     arc_r=1.18, arc_lw=6, label_r=1.34, fontsize=9):
    half_gap = (2 * np.pi / len(angles)) * 0.38
    family_angles = {}
    for label, angle in zip(dim_labels, angles):
        model = axis_model.get(label, "")
        color = model_colors.get(model, "#AAAAAA")
        theta = np.linspace(angle - half_gap, angle + half_gap, 40)
        ax.plot(theta, np.full_like(theta, arc_r), color=color, lw=arc_lw,
                solid_capstyle="round", clip_on=False, zorder=10)
        family_angles.setdefault(model, []).append(angle)
    for model, ang_list in family_angles.items():
        mid = np.arctan2(np.mean(np.sin(ang_list)), np.mean(np.cos(ang_list)))
        color = model_colors.get(model, "#AAAAAA")
        ax.text(mid, label_r, model, ha="center", va="center", fontsize=fontsize,
                fontweight="bold", color=color, clip_on=False, zorder=11,
                bbox=dict(boxstyle="round,pad=0.2", fc="white", ec=color, lw=1.2, alpha=0.90))


def draw_section_layers(ax, angles, section_layers, band_inner=1.09, band_outer=1.20):
    step = angles[1] - angles[0]
    half = step / 2
    for sec in section_layers:
        idx = sec["dim_idx"]
        color = sec["color"]
        a0 = angles[idx[0]] - half
        a1 = angles[idx[-1]] + half
        theta = np.linspace(a0, a1, 300)
        ax.fill_between(theta, band_inner, band_outer, color=color, alpha=0.13,
                         clip_on=False, zorder=5)
        ax.plot(theta, np.full_like(theta, band_inner), color=color, lw=0.8,
                alpha=0.40, clip_on=False, zorder=6)
        ax.plot(theta, np.full_like(theta, band_outer), color=color, lw=0.8,
                alpha=0.40, clip_on=False, zorder=6)


def draw_axis_label_boxes(ax, angles, dim_labels, label_r=1.33):
    for label, angle in zip(dim_labels, angles):
        ax.text(angle, label_r, label, ha="center", va="center", fontsize=AXIS_LABEL_FONTSIZE,
                color="#222222", clip_on=False, zorder=25, multialignment="center",
                bbox=dict(boxstyle="round,pad=0.30", fc="white", ec="#BBBBBB", lw=0.9, alpha=0.97))


def compute_iqr_band(sub_df, norm_cols):
    q25 = sub_df[norm_cols].quantile(0.25).to_numpy(dtype=float)
    q75 = sub_df[norm_cols].quantile(0.75).to_numpy(dtype=float)
    mean = sub_df[norm_cols].mean(skipna=True).to_numpy(dtype=float)
    close = lambda arr: np.append(arr, arr[0])
    return close(q25), close(q75), close(mean)


def plot_single_profile(ax, sub_df, norm_cols, angles, profile_name, color, dim_labels,
                         axis_model, model_colors, section_layers,
                         individual_alpha=0.15, individual_lw=0.5, mean_lw=2.8):
    for _, row in sub_df.iterrows():
        v = np.append(row[norm_cols].to_numpy(dtype=float), row[norm_cols].iloc[0])
        ax.plot(angles, v, color=color, alpha=individual_alpha, lw=individual_lw, zorder=1)
    q25, q75, mean_vals = compute_iqr_band(sub_df, norm_cols)
    ax.fill_between(angles, q25, q75, color=color, alpha=0.22, zorder=2)
    ax.plot(angles, q25, color=color, alpha=0.45, lw=1.0, ls="--", zorder=4)
    ax.plot(angles, q75, color=color, alpha=0.45, lw=1.0, ls="--", zorder=4)
    ax.plot(angles, mean_vals, "o-", color=color, lw=mean_lw, ms=5,
            markerfacecolor="white", markeredgecolor=color, markeredgewidth=1.8,
            label=f"{profile_name} mean (n={len(sub_df)})", zorder=5)
    ax.fill(angles, mean_vals, color=color, alpha=0.10, zorder=3)
    draw_model_arcs(ax, angles[:-1], dim_labels, axis_model, model_colors)
    draw_section_layers(ax, angles[:-1], section_layers)
    draw_axis_label_boxes(ax, angles[:-1], dim_labels)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels([""] * len(dim_labels))
    ax.tick_params(axis="x", length=0)
    ax.set_ylim(0, 1)
    ax.set_yticks([0.25, 0.50, 0.75, 1.00])
    ax.set_yticklabels(["0.25", "0.50", "0.75", "1.00"], fontsize=GRID_TICK_FONTSIZE, alpha=0.5, color="#555555")
    ax.spines["polar"].set_color("#272727")
    ax.grid(color="#121111", lw=0.6)

print("Radar chart helpers loaded (structural reuse from notebook 108 -- DS4's own "
      "5 features and k=3 clusters, not DS5's content)")


In [ ]:
def plot_phenotype_profiles_multi(df, cluster_col, save_path=None):
    available = OrderedDict((k, v) for k, v in RADAR_DIMS.items() if v in df.columns)
    dim_labels = list(available.keys())
    dim_feats = list(available.values())
    am = {k: AXIS_MODEL[k] for k in dim_labels}

    norm_map = normalize_features(df, dim_feats)
    radar_df = df.copy()
    for feat in dim_feats:
        radar_df[f"{feat}_norm"] = norm_map[feat]
    norm_cols = [f"{feat}_norm" for feat in dim_feats]

    angles = np.linspace(0, 2 * np.pi, len(dim_labels), endpoint=False)
    angles = np.append(angles, angles[0])

    clusters = sorted(df[cluster_col].unique())
    k = len(clusters)
    fig, axes = plt.subplots(1, k, figsize=(9 * k, 9.5), subplot_kw=dict(polar=True))
    if k == 1:
        axes = [axes]
    # bottom=0.42 (up from 0.15) and legend fontsize 18 (up from 10-11) --
    # verified visually against overlap with the outer panels' axis labels;
    # see scripts/regenerate_ds4_phenotype_radar.py.
    fig.subplots_adjust(wspace=0.6, left=0.05, right=0.95, top=0.83, bottom=0.42)

    for c, ax in zip(clusters, axes):
        sub = radar_df[radar_df[cluster_col] == c].copy()
        color = CLUSTER_COLORS.get(c, "#888888")
        if sub.empty:
            ax.set_title(f"cluster {c}\n(no data)")
            continue
        plot_single_profile(ax=ax, sub_df=sub, norm_cols=norm_cols, angles=angles,
                             profile_name=f"cluster {c}", color=color, dim_labels=dim_labels,
                             axis_model=am, model_colors=MODEL_COLORS,
                             section_layers=SECTION_LAYERS)

    profile_patches = [
        mpatches.Patch(facecolor=CLUSTER_COLORS.get(c, "#888888"),
                       label=f"cluster {c} mean (n={(df[cluster_col]==c).sum()})")
        for c in clusters
    ]
    profile_patches.append(mpatches.Patch(facecolor="grey", alpha=0.35,
                                           label="IQR band (25th-75th pct.)"))
    leg0 = fig.legend(handles=profile_patches, loc="upper center", bbox_to_anchor=(0.5, 1.03),
                       ncol=1, fontsize=18, framealpha=0.95, edgecolor="#CCCCCC",
                       title="Clusters (mean line + IQR)", title_fontsize=18)
    fig.add_artist(leg0)

    def _desc(m):
        return {"InT": "Information Theory", "StoP": "Stochastic Processes",
                "DyS": "Dynamical Systems", "NeT": "Network Theory",
                "StM": "Statistical Mechanics"}.get(m, m)

    arc_patches = [mpatches.Patch(color=col, label=f"{m} -- {_desc(m)}")
                   for m, col in MODEL_COLORS.items()]
    leg1 = fig.legend(handles=arc_patches, loc="lower left", bbox_to_anchor=(0.02, 0.02),
                       ncol=1, fontsize=18, framealpha=0.95, edgecolor="#CCCCCC",
                       title="Model families (axis color)", title_fontsize=18)
    fig.add_artist(leg1)

    sec_patches = [mpatches.Patch(color=s["color"], alpha=0.55, label=s["name"])
                   for s in SECTION_LAYERS]
    fig.legend(handles=sec_patches, loc="lower right", bbox_to_anchor=(0.98, 0.02),
               ncol=1, fontsize=18, framealpha=0.95, edgecolor="#CCCCCC",
               title="Conceptual layers", title_fontsize=18)

    if save_path is not None:
        fig.savefig(save_path, bbox_inches="tight", dpi=200)
    plt.close(fig)
    return fig, axes


plot_phenotype_profiles_multi(
    complete, cluster_col="cluster",
    save_path=FIG_DIR / "fig5_ds4_phenotype_radar.png",
)
print("Wrote fig5_ds4_phenotype_radar.png")


## Interpretation and limits

**Read the cells above for live numbers.**

- This is a Phase 1 pass only -- one clustering solution on 5 curated, redundancy-checked features. Whether further paths (mirroring DS5's six-path exploration) are warranted is a decision for *after* this notebook's real result, not made in advance.
- **Hierarchical clustering (ward) triangulation (cell 9b):** compares against the k-means solution via ARI -- a high value means k=3 is not an artifact of that one algorithm (matches DS5 Phase 1's own multi-method practice).
- **Radar chart (Figure 5):** adapted from `notebooks/108_ds5_balanced_profile_radar.ipynb`'s plotting machinery -- structural reuse only, DS4's own 5 features and 3 clusters, not DS5's content.
- Demographic overlay is exploratory/secondary (N=283 subset, no Likert scales) -- do not expect DS5-level psychosocial interpretability.
- `tau_hat` and `chi_OU` inherit the daily/weekly timescale ambiguity flagged throughout DS4 Step 1 -- these clusters describe daily-timescale dynamics specifically.
- No experimental condition in DS4 -- clusters describe naturalistic heterogeneity, not a friction-response typology.

### DS4 weakness flags

FLAG I (complete-case attrition confound), FLAG II (no condition), FLAG III (thin demographic overlay), FLAG IV (daily-timescale specific) -- see `ds4_phenotyping_blueprint.md` SS6 for full text.


## Forward connections

| Connection | Status |
|---|---|
| Further DS4 Step 2 paths (condition-free analogs of DS5's exploration) | Pending -- decide based on this notebook's real result |
| DS4 vs DS5 phenotype-structure comparison | Candidate future synthesis, not yet started |
| DS4 Step 3 (personalization) | Not started -- Step 2 is not yet exhausted |
